In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

In [2]:
# EXTRACT: Read the raw fuel price data
df = pd.read_csv("raw_fuel_prices.csv")

print("=== RAW DATA ===")
df

=== RAW DATA ===


,date,petrol_95,diesel
0,2024-01-03,22.87,21.42
1,2024-02-07,23.11,21.68
2,2024-03-06,23.49,22.05
3,2024-04-03,23.82,22.41
4,2024-05-08,24.15,22.78
5,2024-06-05,24.48,23.14
6,2024-07-03,24.72,23.39
7,2024-08-07,24.95,23.63
8,2024-09-04,25.18,23.87
9,2024-10-02,25.41,24.11


In [3]:
# Check data types and basic info
print("=== DATA INFO ===")
df.info() # Checks Datatypes, any missing value, how much memory

print("\n=== BASIC STATISTICS ===")
df.describe() # Checks the average price, min and max.


=== DATA INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date       17 non-null     object 
 1   petrol_95  17 non-null     float64
 2   diesel     17 non-null     float64
dtypes: float64(2), object(1)
memory usage: 536.0+ bytes

=== BASIC STATISTICS ===


,petrol_95,diesel
count,17.000000,17.000000
mean,25.087647,23.762941
std,1.293945,1.364898
min,22.870000,21.420000
25%,24.150000,22.780000
50%,25.180000,23.870000
75%,26.100000,24.830000
max,27.020000,25.790000


In [4]:
# TRANSFORM: Convert date from text to proper datetime
df['date'] = pd.to_datetime(df['date'])

# Add useful columns for analysis
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

print("=== AFTER TRANSFORM ===")
print(df.dtypes)

print("\n=== DATA WITH NEW COLUMNS ===")

=== AFTER TRANSFORM ===
date         datetime64[ns]
petrol_95           float64
diesel              float64
year                  int32
month                 int32
dtype: object

=== DATA WITH NEW COLUMNS ===


In [ ]:
# Plot petrol and diesel prices over time

# It creates blank canvas for chart.
plt.figure(figsize=(10, 5)) # it means the canvas is 10 inches and 5 inches tall
plt.plot(df['date'], df['petrol_95'], label='Petrol 95', marker='o') # it draws the 1st line on chart
plt.plot(df['date'], df['diesel'], label='Diesel', marker='s')

plt.title('South African Fuel Prices Over Time') # Adds title of the graph
plt.xlabel('Date') # Labels the Horizontal axis as "Date"
plt.ylabel('Price (Rands)') # Labels the the Vertical axis as "Price (Rands)"
plt.legend() # Shows the legend box which line is Petrol 95 and which is Diesel.
plt.grid(True) # Adds light gray grid lines behind your chart.
plt.xticks(rotation=45) # It rotates the labels on the X axis by 45 degrees.
plt.tight_layout() # Automatiacally adjusts the chart so nothing gets cut off.


plt.show()


In [ ]:
# LOAD: Save clean data to SQLite database
conn = sqlite3.connect("fuel_prices.db")

# Write the dataframe to a table called 'fuel_prices'
df.to_sql("fuel_prices", conn, if_exists="replace", index=False)

print("=== LOADED TO DATABASE ===")
print("Saved to fuel_prices.db")

df_from_db = pd.read_sql("SELECT * FROM fuel_prices", conn)
print(f"\nRead back {len(df_from_db)} rows from database")
print(df_from_db.head())

conn.close()

In [ ]:
# Reconnect to the database
conn = sqlite3.connect("fuel_prices.db")

# Query 1: Average petrol price by year
query = """
SELECT year, AVG(petrol_95) as avg_petrol
FROM fuel_prices
GROUP BY year
"""

avg_by_year = pd.read_sql(query, conn)

print("=== AVERAGE PETROL PRICE BY YEAR ===")
print(avg_by_year)

conn.close()

In [ ]:
conn = sqlite3.connect("fuel_prices.db")

# Find the month with the highest petrol price
query = """
SELECT month, petrol_95
FROM fuel_prices
WHERE petrol_95 = (SELECT MAX(petrol_95) FROM fuel_prices)
"""

# Option 2 same result
# query = """
# SELECT month, petrol_95
# FROM fuel_prices
# ORDER BY petrol_95 DESC
# LIMIT 1
# """


result = pd.read_sql(query, conn)
print("=== HIGHEST PETROL PRICE ===")
print(result)

conn.close()

In [ ]:
conn = sqlite3.connect("fuel_prices.db")

query = """
SELECT date, petrol_95, (petrol_95 - diesel) as price_gap
FROM fuel_prices
ORDER BY price_gap DESC
"""

result = pd.read_sql(query, conn)
print("=== PETROL vs DIESEL PRICE GAP ===")
print(result)

conn.close()




In [ ]:
conn = sqlite3.connect("fuel_prices.db")

query = """
SELECT 
    date,
    ROUND(petrol_95, 2) as petrol,
    ROUND(diesel, 2) as diesel,
    ROUND((petrol_95 - diesel), 2) as gap
FROM fuel_prices
ORDER BY gap DESC
"""

result = pd.read_sql(query, conn)
print("=== PETROL vs DIESEL PRICE GAP ===")
print(result.to_string())  # Forces all columns to show

conn.close()